In [47]:
import sqlite3
import pandas as pd
import json

# Caminho para seu banco .sqlite
db_path = r"C:\Users\fabio\Documents\GitHub\Easy-Router-Machine\data\processed\streets\streets.sqlite"

# Caminho para o .so ou .dll do SpatiaLite
#mod_spatialite_path = "/usr/lib/mod_spatialite.so"  # Linux
mod_spatialite_path = "mod_spatialite.dll"  # Windows

In [48]:
# Conectar no BD
conn = sqlite3.connect(db_path)

In [49]:
#CARREGAR A EXTENSÃO 
conn.enable_load_extension(True)
conn.load_extension(mod_spatialite_path)
cur = conn.cursor()



In [ ]:
cur = conn.cursor()

# Listar tabelas do banco
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
tabelas = cur.fetchall()

for tabela in tabelas:
    print(tabela[0])

In [50]:
conn.execute('SELECT load_extension("mod_spatialite.dll")')

In [ ]:


# Consulta SQL com a função AsGeoJSON()
query = """
-- https://www.gaia-gis.it/fossil/libspatialite/wiki?name=VirtualRouting
WITH vars AS (
    SELECT 
        -16.7802859 AS lat_o, 	-- latitude de origem
        -49.2717158 AS long_o, 	-- longitude de origem

        -16.803097 AS lat_d,  	-- latitude de destino
        -49.205362 AS long_d,   -- longitude de destino

        0.5 AS Box_LatLong      -- Usado para realizar filtro, se diminir aumenta velocidade mas pode não localizar ponto
),
origem AS (
    SELECT node_id as Node_From
    FROM (
        SELECT node_id, 
               ST_Distance(ST_Point(long_o, lat_o), geometry) AS dist
        FROM roads_nodes, vars
        WHERE
                X(geometry) >= long_o   - Box_LatLong AND X(geometry) <= long_o + Box_LatLong
            AND Y(geometry) >= lat_o    - Box_LatLong AND Y(geometry) <= lat_o  + Box_LatLong
        ORDER BY dist ASC
        LIMIT 1
    )
),
destino AS (
    SELECT node_id as Node_To
    FROM (
        SELECT node_id, 
               ST_Distance(ST_Point(long_d, lat_d), geometry) AS dist
        FROM roads_nodes, vars
        WHERE
                X(geometry) >= long_d   - Box_LatLong AND X(geometry) <= long_d + Box_LatLong
            AND Y(geometry) >= lat_d    - Box_LatLong AND Y(geometry) <= lat_d  + Box_LatLong
        ORDER BY dist ASC
        LIMIT 1
    )
)
SELECT *, AsGeoJSON(nc.Geometry) AS geometry_geojson
FROM router_time nc, origem o, destino d
WHERE
	NodeFrom = o.Node_From AND NodeTo = d.Node_To
"""

cur.execute(query)
rows = cur.fetchall()

columns = [desc[0] for desc in cur.description]
print(columns)
df = pd.DataFrame(rows, columns=columns)


#cur.close()
#conn.close()


['Algorithm', 'ArcRowid', 'NodeFrom', 'NodeTo', 'Cost', 'Geometry', 'Name', 'Node_From', 'Node_To', 'geometry_geojson']


In [ ]:
geojson_str = df['geometry_geojson'].dropna().iloc[0]
print(geojson_str)

{"type":"LineString","coordinates":[[-49.2716957,-16.7804623],[-49.2718569,-16.7809942],[-49.2720224,-16.7815426],[-49.2722255,-16.7822159],[-49.2722664,-16.782205],[-49.2724818,-16.7821511],[-49.2726953,-16.7828092],[-49.2723883,-16.7828946],[-49.2726414,-16.7835505],[-49.2714596,-16.7839046],[-49.2711678,-16.783987],[-49.2704733,-16.784178],[-49.2703442,-16.7842132],[-49.2702718,-16.7842319],[-49.2685078,-16.7847371],[-49.2680957,-16.7848414],[-49.2674133,-16.7849162],[-49.2667999,-16.7849488],[-49.2650461,-16.7850817],[-49.2633428,-16.7852112],[-49.2606066,-16.7854188],[-49.2595108,-16.7855124],[-49.2583465,-16.7856218],[-49.2577002,-16.7856627],[-49.2569447,-16.7857127],[-49.2562232,-16.7857717],[-49.2551302,-16.7858449],[-49.2542944,-16.7859052],[-49.2541342,-16.7859181],[-49.2534165,-16.7859761],[-49.2531521,-16.7859974],[-49.2512526,-16.7861606],[-49.250813,-16.7861986],[-49.2489335,-16.7863372],[-49.2478363,-16.7864319],[-49.2465284,-16.7865325],[-49.2456794,-16.7865886],[-49.2

In [ ]:
geojson_obj = json.loads(geojson_str)

In [67]:
geojson_binario = df['Geometry'].dropna().iloc[0]


In [85]:
import random

def gerar_coords_goias():
    lat = random.uniform(-18.0, -12.0)
    lon = random.uniform(-51.0, -46.0)
    return lat, lon

In [86]:
def montar_query(lat_o, lon_o, lat_d, lon_d):
    return f"""
    WITH vars AS (
        SELECT 
            {lat_o} AS lat_o,
            {lon_o} AS long_o,
            {lat_d} AS lat_d,
            {lon_d} AS long_d,
            0.5 AS Box_LatLong
    ),
    origem AS (
        SELECT node_id as Node_From
        FROM (
            SELECT node_id,
                   ST_Distance(ST_Point(long_o, lat_o), geometry) AS dist
            FROM roads_nodes, vars
            WHERE
                    X(geometry) >= long_o - Box_LatLong AND X(geometry) <= long_o + Box_LatLong
                AND Y(geometry) >= lat_o - Box_LatLong AND Y(geometry) <= lat_o + Box_LatLong
            ORDER BY dist ASC
            LIMIT 1
        )
    ),
    destino AS (
        SELECT node_id as Node_To
        FROM (
            SELECT node_id,
                   ST_Distance(ST_Point(long_d, lat_d), geometry) AS dist
            FROM roads_nodes, vars
            WHERE
                    X(geometry) >= long_d - Box_LatLong AND X(geometry) <= long_d + Box_LatLong
                AND Y(geometry) >= lat_d - Box_LatLong AND Y(geometry) <= lat_d + Box_LatLong
            ORDER BY dist ASC
            LIMIT 1
        )
    )
    SELECT *, AsGeoJSON(nc.Geometry) AS geometry_geojson
    FROM router_time nc, origem o, destino d
    WHERE NodeFrom = o.Node_From AND NodeTo = d.Node_To
    """

In [99]:
import sqlite3
import pandas as pd
import json

# Caminho para seu banco .sqlite
db_path = r"C:\Users\fabio\Documents\GitHub\Easy-Router-Machine\data\processed\streets\streets.sqlite"

# Caminho para o .so ou .dll do SpatiaLite
#mod_spatialite_path = "/usr/lib/mod_spatialite.so"  # Linux
mod_spatialite_path = "mod_spatialite.dll"  # Windows

# Conectar no BD
conn = sqlite3.connect(db_path)

#CARREGAR A EXTENSÃO 
conn.enable_load_extension(True)
conn.load_extension(mod_spatialite_path)

cur = conn.cursor()

conn.execute('SELECT load_extension("mod_spatialite.dll")')

In [ ]:
import sqlite3



def executar_rota(i):
    lat_o, lon_o = gerar_coords_goias()
    lat_d, lon_d = gerar_coords_goias()
    query = montar_query(lat_o, lon_o, lat_d, lon_d)
    try:
        db_path = r"C:\Users\fabio\Documents\GitHub\Easy-Router-Machine\data\processed\streets\streets.sqlite"
        mod_spatialite_path = "mod_spatialite.dll"  # Windows
        # Cada thread cria sua própria conexão
        conn = sqlite3.connect(db_path, check_same_thread=False)
        conn.enable_load_extension(True)
        conn.load_extension(mod_spatialite_path)
        conn.execute('SELECT load_extension("mod_spatialite.dll")')
        cur = conn.cursor()
        cur.execute(query)
        rows = cur.fetchall()
        conn.close()
        columns = [desc[0] for desc in cur.description]
        df = pd.DataFrame(rows, columns=columns)
        print(f"[{i}] Rota executada com sucesso.")
        geojson_obj = json.loads(df['geometry_geojson'].dropna().iloc[0])
        return  geojson_obj
    except Exception as e:
        print(f"[{i}] ERRO: {e}")

In [114]:
ROTA = executar_rota(1)
print(ROTA)

[1] Rota executada com sucesso.
{'type': 'LineString', 'coordinates': [[-47.0422876, -17.8447168], [-47.0419925, -17.8447704], [-47.0418692, -17.8447985], [-47.04179, -17.8448534], [-47.0417364, -17.8449249], [-47.0416868, -17.8450666], [-47.0416331, -17.8452734], [-47.0415929, -17.8455759], [-47.0415446, -17.8460904], [-47.0415205, -17.8467197], [-47.0414695, -17.8477231], [-47.0413783, -17.8493239], [-47.0413381, -17.850072], [-47.0412871, -17.8509949], [-47.0412388, -17.8519919], [-47.0412134, -17.8525829], [-47.0412201, -17.8528025], [-47.0412415, -17.8530961], [-47.0413944, -17.8544466], [-47.0415171, -17.8556574], [-47.0416211, -17.856558], [-47.041668, -17.8569614], [-47.0418853, -17.8589527], [-47.0420086, -17.8601117], [-47.042132, -17.8612899], [-47.0422688, -17.8622524], [-47.0423962, -17.863419], [-47.0424968, -17.8643011], [-47.0426027, -17.8653171], [-47.0426316, -17.8656847], [-47.042638, -17.8658462], [-47.0426872, -17.8658609], [-47.0427865, -17.8658979], [-47.0429233,

In [102]:
import concurrent.futures
import time

def teste_sobrecarga(num_threads=50):
    inicio = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(executar_rota, i) for i in range(num_threads)]
        concurrent.futures.wait(futures)
    fim = time.time()
    print(f"Teste com {num_threads} threads finalizado em {fim - inicio:.2f} segundos.")


In [104]:
teste_sobrecarga(5)

Teste com 5 threads finalizado em 0.00 segundos.
